In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from helpers.utils import f2_score, set_seed
import yaml
from ultralytics import YOLO
set_seed(42)

In [2]:
DATA = PROJECT_ROOT / "data"
DATA_YAML = DATA / "dataset.yaml"
START = PROJECT_ROOT / "models" / "runs" / "yolo26n_unfreeze50" / "weights" / "best.pt"
RUNS = PROJECT_ROOT / "models" / "runs"

In [3]:
HYP = PROJECT_ROOT / "models" / "runs" / "yolo26n_tune" / "best_hyperparameters.yaml"
with open(HYP, "r") as f:
    hyp = yaml.safe_load(f)

In [4]:
model = YOLO(str(START))
results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    seed=42,
    optimizer="AdamW",
    lr0=hyp["lr0"],
    lrf=hyp["lrf"],
    momentum=hyp["momentum"],
    weight_decay=hyp["weight_decay"],
    warmup_epochs=hyp["warmup_epochs"],
    warmup_momentum=hyp["warmup_momentum"],
    cfg=str(HYP),
    project=str(RUNS),
    name="yolo26n_tuned50_fixed",
    exist_ok=True,
)

New https://pypi.org/project/ultralytics/8.4.116 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.56  Python-3.11.15 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.00674, box=7.55674, cache=False, cfg=C:\Users\Arda\Desktop\Engineering\early-fire-detection\models\runs\yolo26n_tune\best_hyperparameters.yaml, classes=None, close_mosaic=5, cls=0.47149, cls_pw=0.00186, compile=False, conf=None, copy_paste=0.00501, copy_paste_mode=flip, cos_lr=False, cutmix=0.00151, data=C:\Users\Arda\Desktop\Engineering\early-fire-detection\data\dataset.yaml, degrees=0.01988, deterministic=True, device=0, dfl=1.59122, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.57591, flipud=0.002, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.00681, hsv_s=0.77053, hsv_v=0.26708, 

In [5]:
tuned = YOLO(str(RUNS / "yolo26n_tuned50_fixed" / "weights" / "best.pt"))
m = tuned.val(data=str(DATA_YAML), split="val", project=str(RUNS), exist_ok=True)
p, r = float(m.box.mp), float(m.box.mr)
print(f"tuned50_fixed  P={p} R={r} mAP50={m.box.map50} F2={f2_score(p, r)}")

Ultralytics 8.4.56  Python-3.11.15 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1860.9578.2 MB/s, size: 248.5 KB)
val: Scanning C:\Users\Arda\Desktop\Engineering\early-fire-detection\data\val\labels.cache... 1300 images, 33 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1300/1300  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 82/82 10.9it/s 7.5s.1ss
                   all       1300       1580      0.929      0.899      0.941      0.641
                  fire        894        963      0.939      0.922      0.953      0.681
                 smoke        574        617      0.918      0.875      0.929      0.601
Speed: 1.5ms preprocess, 1.8ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to C:\Users\Arda\Desktop\Engineering\early-fire-detection\models\

First tuned run ignored the tuned lr0/momentum because optimizer was auto. We retrained with AdamW + tuned lr0/momentum tuned50_fixed.

Val F2: 0.921 (old) vs 0.904 (fixed). mAP50 stayed almost the same, but recall dropped.

Since we care more about recall/F2, we keep the old yolo26n_tuned50 as the final model.